# 欢迎来到你的第一份作业！

下面是练习说明。请先自己动手试一试；卡住时可以查看 solutions 文件夹，或随时提问。


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">在开始作业之前 ——</h2>
            <span style="color:#f71;">先收藏这门课的资源页：包含全部幻灯片链接，以及后续会持续补充的资料。<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            建议保持书签，后面还会往这里加有用链接。
            </span>
        </td>
    </tr>
</table>


# 家庭作业练习

把第 1 天的「网页摘要」项目升级：改用通过 **Ollama** 在本地跑的开源模型，而不是 OpenAI。

后续项目若不想付费调用 API，都可以沿用这套本地调用方式。

**好处：**
1. 无 API 费用 —— 开源本地模型
2. 数据不出你的机器

**代价：**
1. 能力明显弱于前沿（Frontier）云端大模型

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 + 清洗 | `Website` + BeautifulSoup |
| Chat Completions `messages` | system / user 拼摘要提示 |
| 本地 Ollama | `http://localhost:11434` |
| 三种调用方式 | 裸 HTTP / `ollama` 包 / OpenAI 兼容 `base_url` |

## Ollama 安装回顾

访问 [ollama.com](https://ollama.com) 安装即可。

装好后，本机 Ollama 服务通常已在跑。打开：  
[http://localhost:11434/](http://localhost:11434/)

应看到 `Ollama is running`。

若没有：新开 Terminal（Mac）或 Powershell（Windows），执行 `ollama serve`；  
另开一个窗口执行 `ollama pull llama3.2`；再刷新上面的地址。

若本机偏慢，可改用 `llama3.2:1b`：先 `ollama pull llama3.2:1b`，再把下面代码里的 `MODEL = "llama3.2"` 改成 `MODEL = "llama3.2:1b"`。


In [ ]:
# ========== 导入：抓网页 + 在笔记本里显示 Markdown ==========

# 导入 requests：用 HTTP 请求本地 Ollama / 抓取网页
import requests
# 从 bs4 导入 BeautifulSoup：解析 HTML，抽出标题与正文
from bs4 import BeautifulSoup
# 从 IPython.display 导入展示工具：在笔记本里渲染 Markdown
from IPython.display import Markdown, display


In [ ]:
# ========== 常量：Ollama 原生聊天 API 与模型名 ==========

# Ollama 原生 /api/chat 地址（不是 OpenAI 兼容的 /v1）
OLLAMA_API = "http://localhost:11434/api/chat"
# JSON 请求头：告诉服务端 body 是 application/json
HEADERS = {"Content-Type": "application/json"}
# 本地模型名：需事先 ollama pull；字符串必须和本机已安装的模型名一致
MODEL = "llama3.2"


In [ ]:
# ========== 构造 messages：与 OpenAI Chat Completions 相同的 role/content 结构 ==========

# 创建 messages 列表：这里只用 user 角色；content 是发给模型的英文提示（可运行字符串不翻译）
messages = [
    {"role": "user", "content": "Describe some of the business applications of Generative AI"}
]


In [ ]:
# ========== 组装 POST 到 Ollama /api/chat 的 JSON payload ==========

# model / messages / stream：stream=False 表示等整段答完再返回（非流式）
payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False
    }


In [ ]:
# ========== 确保本地已拉取 llama3.2（魔法命令调 shell） ==========

# 笔记本 ! 前缀：在 shell 里执行 ollama pull；已存在则会很快结束
!ollama pull llama3.2


In [ ]:
# ========== 路径 A：用 requests 直连 Ollama 原生 HTTP API ==========

# 若本格失败：可试后面两格的替代写法；并复查顶部「Ollama 安装回顾」
# 仍不行再联系课程助教 / 讲师

# POST 到 OLLAMA_API；json=payload 自动序列化；headers 声明 JSON
response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
# Ollama 非流式响应：['message']['content'] 即助手回复正文
print(response.json()['message']['content'])


# 引入 ollama Python 包

下面做同一件事，但改用更优雅的 `ollama` Python 包，而不是手写 HTTP。

底层仍然请求本机 `localhost:11434` 上的 Ollama 服务，与上一格等价。


In [ ]:
# ========== 路径 B：用 ollama 官方 Python 包聊天 ==========

# 导入 ollama 包：封装对本地 Ollama 的调用
import ollama

# chat：传入 model 与 messages；返回结构里仍有 message.content
response = ollama.chat(model=MODEL, messages=messages)
# 打印助手回复正文
print(response['message']['content'])


## 替代路径：用 OpenAI Python 库连接 Ollama


In [ ]:
# ========== 路径 C：OpenAI 客户端 + base_url 指向本地 Ollama /v1 ==========

# 另一种常见写法：复用 OpenAI 客户端库去调 Ollama（OpenAI-compatible API）

# 从 openai 导入 OpenAI 客户端类
from openai import OpenAI
# base_url 指向本地兼容端点；api_key 对本地 Ollama 通常任意非空即可
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

# 与云端相同的 chat.completions.create 调用面；实际打到本机 Ollama
response = ollama_via_openai.chat.completions.create(
    model=MODEL,
    messages=messages
)

# choices[0].message.content：标准 Chat Completions 响应路径
print(response.choices[0].message.content)


## 为什么「用 OpenAI 代码调 Ollama」也能工作？

看起来很怪，对吧？我们明明写的是 OpenAI 的客户端代码，却打到了 Ollama。发生了什么？

要点如下：

Python 类 `OpenAI` 本质是 OpenAI 工程师写的**客户端库（client library）**：在你电脑上发 HTTP 请求。

当你调用 `openai.chat.completions.create()` 时，默认会请求：  
`https://api.openai.com/v1/chat/completions`

真正的 GPT 算力在 OpenAI 云端，不在你的笔记本上。

因为这套 Web API 太流行，许多其它厂商提供了**相同形状的端点**，以便复用同一套调用方式。

Ollama 在你本机暴露：`http://localhost:11434/v1/chat/completions`  
第 2 周还会看到 Gemini、DeepSeek 等也这么做。

于是 OpenAI 团队扩展了客户端：允许你指定不同的 `base_url`，用同一套库去调任何兼容 API。

所以当你写：  
`ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')`  
请求形状不变，只是目标从 OpenAI 云端换成了本机 Ollama。


## 再试一下出色的推理模型 DeepSeek

这里使用被蒸馏到 **1.5B** 的 DeepSeek-reasoner 版本。  
它实际上是 Qwen 的 1.5B 变体，用 DeepSeek R1 生成的合成数据做了微调（fine-tune）。

其它尺寸见 [ollama.com/library/deepseek-r1](https://ollama.com/library/deepseek-r1)，一直到完整 **671B**（约 404GB 磁盘），对大多数人来说太大了。


In [ ]:
# ========== 拉取本地 DeepSeek R1 1.5B ==========

# shell：下载/更新 deepseek-r1:1.5b 到本机 Ollama 模型库
!ollama pull deepseek-r1:1.5b


In [ ]:
# ========== 用 OpenAI 兼容客户端问 DeepSeek：观察 <think> 推理轨迹 ==========

# 可能跑几分钟！成功时通常先看到 <think>…</think> 里的「思考」，再给出定义

# 复用上一格的 ollama_via_openai；model 换成 deepseek-r1:1.5b
response = ollama_via_openai.chat.completions.create(
    model="deepseek-r1:1.5b",
    # user prompt 保留英文：发给模型的内容不翻译，避免改变行为
    messages=[{"role": "user", "content": "Please give definitions of some core concepts behind LLMs: a neural network, attention and the transformer"}]
)

# 打印完整助手回复（含 thinking 与最终答案）
print(response.choices[0].message.content)


# 现在轮到你做练习

把第 1 天的代码接到这里：做一个**网站摘要器**，用本地跑的 Llama 3.2（或你上面选的任一路径）代替 OpenAI。


1. 创建 `Website` 类


In [16]:
# ========== 练习导入：环境变量 + 抓取 + OpenAI 兼容客户端 + 展示 ==========

# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量（本练习主要用本地 Ollama，仍可能预留）
from dotenv import load_dotenv
# BeautifulSoup：解析 HTML
from bs4 import BeautifulSoup
# requests：HTTP GET 网页
import requests
# OpenAI 客户端：后面用 base_url 指向本地 Ollama
from openai import OpenAI
# Markdown / display：在笔记本里漂亮展示摘要
from IPython.display import Markdown, display


In [2]:
# ========== Website 类：抓取 URL，清洗 HTML，留下标题与正文 ==========

# 浏览器 User-Agent：降低部分站点把脚本请求直接拒掉的概率
headers = {
 "user-agent": "mozilla/5.0 (windows nt 10.0; win64; x64) applewebkit/537.36 (khtml, like gecko) chrome/117.0.0.0 safari/537.36"
}

class Website:
    def __init__(self, url):
        # 保存原始 URL，便于调试与追溯
        self.url = url
        # GET 网页 HTML；带上 headers 模拟浏览器
        response = requests.get(url, headers = headers)
        # 用 html.parser 解析响应字节内容
        soup = BeautifulSoup(response.content, 'html.parser')
        # 页面 <title>；没有则给占位英文串（可运行字符串保持原样）
        self.title = soup.title if soup.title else "No title found for this website"

        # 删除脚本/样式/图片/输入控件等对摘要无用的节点（decompose 从树中移除）
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 抽出可见文本：换行分隔、strip 掉多余空白
        self.text = soup.body.get_text(separator="\n", strip=True)


In [3]:
# ========== Prompt：system 定角色，user 拼网页标题 + 正文 ==========

# system prompt 保留英文：这是发给模型的指令，改译会改变摘要风格/行为
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

def getUserPrompt(website):
    # 先告诉模型正在看哪个标题（title 可能是 Tag，原逻辑保持）
    userPrompt = f"You are looking at a website titled {website.title}"
    # 再要求短摘要；若有新闻/公告一并概括（英文指令不翻译）
    userPrompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    # 拼上清洗后的网页正文，作为模型的主要阅读材料
    userPrompt += website.text
    return userPrompt


In [4]:
# ========== 把 system + user 收成 Chat Completions 的 messages 列表 ==========

def getPromptMessageFor(website):
    # 标准两段式：system 定规则，user 放具体网页内容
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": getUserPrompt(website)}
    ]


In [8]:
# ========== summarize：抓站 → 本地 Ollama（DeepSeek）→ 返回摘要文本 ==========

def summarize(url):
    # 实例化 Website：内部完成抓取与清洗
    website = Website(url)
    # OpenAI 兼容客户端指向本机 Ollama /v1；api_key 占位即可
    ollamaAi = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
    # 注意：此处作者选用 deepseek-r1:1.5b（不是前面常量 MODEL 的 llama3.2）
    response = ollamaAi.chat.completions.create(
        model = "deepseek-r1:1.5b",
        messages = getPromptMessageFor(website)
    )
    # 取出助手消息正文
    return response.choices[0].message.content


In [14]:
# ========== display_summary：摘要 + 在笔记本里渲染 Markdown ==========

def display_summary(url):
    # 调用 summarize 拿到模型生成的 Markdown 字符串
    summary = summarize(url)
    # display(Markdown(...))：比裸 print 更适合阅读标题/列表
    display(Markdown(summary))


In [ ]:
# ========== 试跑：对讲师个人站做一次本地模型摘要 ==========

# 可改成你感兴趣的 URL；需本机 Ollama 已 pull deepseek-r1:1.5b 且服务在跑
display_summary("https://edwarddonner.com")
